In [ ]:
# Notebook 16e: CNN Head — Real Only (path_mnist)
import os, numpy as np, pandas as pd, gc
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from sklearn.metrics import classification_report
SEED=42; tf.random.set_seed(SEED); np.random.seed(SEED)
CLASSES=9; BATCH_SIZE=128; EPOCHS=50
CLASS_NAMES=['Adipose', 'Background', 'Debris', 'Lymphocytes', 'Mucus', 'Smooth Muscle', 'Normal Colon Mucosa', 'Cancer-Assoc. Stroma', 'Colorectal Adenocarcinoma']
ACTIVATION='relu'; DROPOUT=0.5
PROJECT_DIR='/content/drive/MyDrive/thesis_project'
DDIM_FEAT=os.path.join(PROJECT_DIR,'data','extracted_features','seed_42','ddim')
REAL_FEAT=os.path.join(PROJECT_DIR,'data','extracted_features','seed_42','real')
SYNTH_NPZ=os.path.join(PROJECT_DIR,'data','synthetic','seed_42','ddim','path_mnist/path_mnist_synthetic.npz')
REAL_NPZ=os.path.join(PROJECT_DIR,'data','processed','seed_42','path_mnist.npz')
RESULTS_DIR=os.path.join(PROJECT_DIR,'results','cnn_head','path_mnist','real_only')
os.makedirs(RESULTS_DIR,exist_ok=True)

In [ ]:
F_tr=np.concatenate([
    np.load(os.path.join(REAL_FEAT,'path_mnist_x_train_features.npy')),
    np.load(os.path.join(REAL_FEAT,'path_mnist_x_val_features.npy'))],axis=0)
real=np.load(REAL_NPZ)
y_tr=np.concatenate([real['y_train'],real['y_val']],axis=0)
F_test=np.load(os.path.join(REAL_FEAT,'path_mnist_x_test_features.npy'))
y_test=real['y_test']
print(f'F_tr: {F_tr.shape}, F_test: {F_test.shape}')

In [ ]:
tf.keras.backend.clear_session(); gc.collect()
model=models.Sequential([
    layers.Input(shape=(1280,)),
    layers.Dense(64,activation=ACTIVATION),
    layers.Dropout(DROPOUT),
    layers.Dense(CLASSES,activation='softmax'),
],name='cnn_head')
model.compile(optimizer=optimizers.Adam(),
    loss='sparse_categorical_crossentropy',metrics=['accuracy'])
model.summary()

In [ ]:
history=model.fit(F_tr,y_tr,batch_size=BATCH_SIZE,epochs=EPOCHS,verbose=1)
pd.DataFrame(history.history).to_csv(
    os.path.join(RESULTS_DIR,'history.csv'),index_label='epoch')
fig,axes=plt.subplots(1,2,figsize=(12,4))
axes[0].plot(history.history['loss'],label='train loss')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch')
axes[0].legend(); axes[0].grid(True,alpha=0.3)
axes[1].plot(history.history['accuracy'],label='train accuracy')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch')
axes[1].legend(); axes[1].grid(True,alpha=0.3)
plt.suptitle('Training Curves — CNN Head Real Only (path_mnist)',fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR,'training_curves.png'),dpi=150,bbox_inches='tight')
plt.close(); print('Training complete.')

In [ ]:
test_loss,test_acc=model.evaluate(F_test,y_test,verbose=0)
print(f'Test accuracy: {test_acc:.4f} | Test loss: {test_loss:.4f}')
y_prob_eval=model.predict(F_test,verbose=0)
y_pred=np.argmax(y_prob_eval,axis=1)
print(classification_report(y_test,y_pred,target_names=CLASS_NAMES,digits=4))
report_dict = classification_report(y_test, y_pred,
    target_names=CLASS_NAMES, digits=4, output_dict=True)
macro_f1    = round(report_dict['macro avg']['f1-score'], 4)
weighted_f1 = round(report_dict['weighted avg']['f1-score'], 4)
pd.DataFrame(report_dict).transpose().to_csv(
    os.path.join(RESULTS_DIR, 'classification_report.csv'))
pd.DataFrame([{
    'condition':   'real_only',
    'dataset':     'path_mnist',
    'test_acc':    round(test_acc, 4),
    'test_loss': round(test_loss,4),
    'macro_f1':    macro_f1,
    'weighted_f1': weighted_f1,
}]).to_csv(os.path.join(RESULTS_DIR, 'summary.csv'), index=False)

In [ ]:
# ── Final Cell: Save Predictions & Confusion Matrix ──────────────────────────
import seaborn as sns
from sklearn.metrics import confusion_matrix

y_prob = model.predict(F_test, verbose=0)
y_pred = np.argmax(y_prob, axis=1)

np.save(os.path.join(RESULTS_DIR, 'y_pred.npy'), y_pred)
np.save(os.path.join(RESULTS_DIR, 'y_test.npy'), y_test)
np.save(os.path.join(RESULTS_DIR, 'y_prob.npy'), y_prob)

cm = confusion_matrix(y_test, y_pred)
pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(
    os.path.join(RESULTS_DIR, 'confusion_matrix.csv'))
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Confusion Matrix — path_mnist | real_only')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.close()
print('All outputs saved.')